In [1]:
from openai import AsyncOpenAI
from agents import Agent, OpenAIResponsesModel, Runner, ModelSettings, ModelRetrySettings
from loguru import logger

MODEL = "gpt-5.3-codex"
BASE_URL = "https://api.tokenlab.sh/v1"
API_KEY = "sk-aeRemEo2sD0YgQWEFGjipWrzTp4LVFUVzHD8bD5fx5PoLMGF"


client = AsyncOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    timeout=90.0,
    max_retries=0,
)

model = OpenAIResponsesModel(
    model=MODEL,
    openai_client=client,
)



In [7]:
from agents import WebSearchTool


agent = Agent(                                                                                                                 
    name="Assistant",                                                                                                          
    instructions="You are a helpful assistant.",                                                                               
    model=model,
    tools=[
        WebSearchTool()
    ],
    model_settings=ModelSettings(
            parallel_tool_calls=False,
            truncation="auto",
            store=False,
            context_management=[{"type": "compaction", "compact_threshold": 200000}],
            prompt_cache_retention="24h",
            response_include=["web_search_call.action.sources"],
            retry=ModelRetrySettings(
                max_retries=3
            )
        ),                                                                                                         
)                                                                                                                              
                                                                                                                                
result = await Runner.run(agent, "meta比较火的广告素材有哪些")    

# print(result.final_output)                                                             
# print(result.final_output)  
for item in result.new_items: 
    print(item)
    print("\n"*10)

OPENAI_API_KEY is not set, skipping trace export


ToolCallItem(agent=Agent(name='Assistant', handoff_description=None, tools=[WebSearchTool(user_location=None, filters=None, search_context_size='medium', external_web_access=None)], mcp_servers=[], mcp_config={}, instructions='You are a helpful assistant.', prompt=None, handoffs=[], model=<agents.models.openai_responses.OpenAIResponsesModel object at 0x11e6d0f50>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=False, truncation='auto', max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=False, prompt_cache_retention='24h', include_usage=None, response_include=['web_search_call.action.sources'], top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=ModelRetrySettings(max_retries=3, backoff=None), context_management=[{'type': 'compaction', 'compact_threshold': 200000}]), input_guardrails=[], output_guardrails=[], output_type=None, h

OPENAI_API_KEY is not set, skipping trace export


In [10]:

import asyncio
import json
import os
from typing import Annotated

from openai import AsyncOpenAI
from agents import Agent, ModelSettings, Runner, function_tool
from agents.models.openai_responses import OpenAIResponsesModel

MODEL = "gpt-5.3-codex"
BASE_URL = "https://api.tokenlab.sh/v1"
API_KEY = "sk-aeRemEo2sD0YgQWEFGjipWrzTp4LVFUVzHD8bD5fx5PoLMGF"


# === 业务场景：本地运行时工具 ===

@function_tool
def fetch_project_brief(
    project_id: Annotated[str, "项目 ID，例如 P001"],
) -> str:
    """从 backend 或本地读取项目简介和营销目标。"""
    # 模拟从 backend API 或本地文件读取
    mock_data = {
        "P001": "游戏：传奇手游；目标：拉新；预算：10万美元/月；地区：北美",
        "P002": "游戏：卡牌 RPG；目标：ROI 优化；预算：5万美元/月；地区：日韩",
    }
    result = mock_data.get(project_id, f"项目 {project_id} 未找到")
    return f"[fetch_project_brief] {result}"


@function_tool
def analyze_material_performance(
    material_id: Annotated[str, "素材 ID，例如 M001"],
) -> str:
    """分析素材投放表现：CTR、转化率、花费等。"""
    # 模拟从本地或 backend metrics 读取
    mock_metrics = {
        "M001": "CTR: 2.3%, CVR: 1.8%, 花费: $1200, ROI: 3.5",
        "M002": "CTR: 1.9%, CVR: 1.2%, 花费: $800, ROI: 2.1",
    }
    result = mock_metrics.get(material_id, f"素材 {material_id} 无数据")
    return f"[analyze_material_performance] {result}"


@function_tool
def generate_campaign_report(
    project_id: Annotated[str, "项目 ID"],
    report_type: Annotated[str, "报告类型：daily/weekly/monthly"],
) -> str:
    """生成投放报告（模拟写入本地文件或返回报告链接）。"""
    # 模拟生成报告文件
    report_path = f"/Users/zhangtianzhu/Project/huya/ANIFORCE/notebooks/03-runtime/aniforce_reports/{project_id}_{report_type}_report.md"
    return f"[generate_campaign_report] 报告已生成: {report_path}"


client = AsyncOpenAI(
        api_key=API_KEY,
        base_url=BASE_URL,
        timeout=90.0,
        max_retries=0,
    )

model = OpenAIResponsesModel(
    model=MODEL,
    openai_client=client,
)

agent = Agent(
    name="ANIFORCE Marketing Assistant",
    instructions=(
        "你是 ANIFORCE 游戏营销平台的助手。"
        "用户会问项目、素材、报告相关问题，你可以调用本地工具查询或生成。"
        "回答简洁、有条理。"
    ),
    model=model,
    tools=[
        fetch_project_brief,
        analyze_material_performance,
        generate_campaign_report,
    ],
    model_settings=ModelSettings(
        parallel_tool_calls=False,
        truncation="auto",
        store=False,
        prompt_cache_retention="24h",
    ),
)

prompt = "帮我查一下项目 P001 的简介，然后分析素材 M001 的表现，生成一份正式的投放报告文件"

result = await Runner.run(agent, prompt, max_turns=5)

print("\n" + "=" * 80)
print("调试输出：result.new_items")
print("=" * 80 + "\n")

for item in result.new_items:
    print(item)
    print("\n" + "-" * 80 + "\n")

print("\n" + "=" * 80)
print("最终回答：")
print("=" * 80 + "\n")
print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export



调试输出：result.new_items

ToolCallItem(agent=Agent(name='ANIFORCE Marketing Assistant', handoff_description=None, tools=[FunctionTool(name='fetch_project_brief', description='从 backend 或本地读取项目简介和营销目标。', params_json_schema={'properties': {'project_id': {'description': '项目 ID，例如 P001', 'title': 'Project Id', 'type': 'string'}}, 'required': ['project_id'], 'title': 'fetch_project_brief_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x11fd5f910>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None), FunctionTool(name='analyze_material_performance', description='分析素材投放表现：CTR、转化率、花费等。', params_json_schema={'properties': {'material_id': {'description': '素材 ID，例如 M001', 'title': 'Material Id', 'type': 'string'}},

OPENAI_API_KEY is not set, skipping trace export


In [15]:

import asyncio
import base64
from pathlib import Path
from typing import Annotated

from openai import AsyncOpenAI
from agents import Agent, ModelSettings, Runner, RunContextWrapper, FunctionTool, function_tool
from agents.models.openai_responses import OpenAIResponsesModel
from agents.tool import ToolOutputImage, ToolOutputFileContent
from pydantic import BaseModel

MODEL = "gpt-5.3-codex"
BASE_URL = "https://api.tokenlab.sh/v1"
API_KEY = "sk-aeRemEo2sD0YgQWEFGjipWrzTp4LVFUVzHD8bD5fx5PoLMGF"


# === 方式 1：@function_tool 返回图像 ===
@function_tool                                                                                                                          
def get_material_thumbnail(                                                                                                             
    material_id: Annotated[str, "素材 ID"],                                                                                             
) -> ToolOutputImage:                                                                                                                   
    """获取素材缩略图。"""                                                                                                              
    red_png_base64 = "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8z8DwHwAFBQIAX8jx0gAAAABJRU5ErkJggg=="                 
    return ToolOutputImage(
        image_url=f"data:image/png;base64,{red_png_base64}",
        detail="low",
    )                                                                                                                                   
                                                                                                                                        
                                                                                                                                        
@function_tool                                                                                                                          
def download_campaign_report(                                                                                                           
    project_id: Annotated[str, "项目 ID"],                                                                                              
) -> ToolOutputFileContent:                                                                                                             
    """下载投放报告文件。"""                                                                                                            
    report_content = f"# {project_id} 投放报告\n\n- CTR: 2.5%\n- ROI: 3.2\n"                                                            
    report_base64 = base64.b64encode(report_content.encode("utf-8")).decode("utf-8")
    return ToolOutputFileContent(
        filename=f"{project_id}_report.md",
        # Responses API 需要 data URL 格式，不是裸 base64。
        file_data=f"data:text/plain;base64,{report_base64}",
    )      


# === 方式 3：自定义 FunctionTool ===

class AnalyzeArgs(BaseModel):
    project_id: str
    date_range: str


async def run_custom_analyze(ctx: RunContextWrapper, args: str) -> str:
    parsed = AnalyzeArgs.model_validate_json(args)
    return f"[custom_analyze] 项目 {parsed.project_id} 在 {parsed.date_range} 的分析完成"


custom_tool = FunctionTool(
    name="custom_analyze",
    description="自定义分析工具：深度分析项目数据",
    params_json_schema=AnalyzeArgs.model_json_schema(),
    on_invoke_tool=run_custom_analyze,
)


client = AsyncOpenAI(api_key=API_KEY, base_url=BASE_URL, timeout=90.0, max_retries=0)
model = OpenAIResponsesModel(model=MODEL, openai_client=client)

agent = Agent(
    name="ANIFORCE Material Assistant",
    instructions="你是 ANIFORCE 助手，可以获取素材缩略图、下载报告、执行分析。",
    model=model,
    tools=[
        get_material_thumbnail,
        download_campaign_report,
        custom_tool,
    ],
    model_settings=ModelSettings(
        parallel_tool_calls=False,
        truncation="auto",
        store=False,
    ),
)

prompt = "帮我获取素材 M001 的缩略图，然后下载项目 P001 的报告"

result = await Runner.run(agent, prompt, max_turns=5)

print("\n" + "=" * 80)
print("调试输出：result.new_items")
print("=" * 80 + "\n")

for item in result.new_items:
    print(item)
    
    print("\n" + "-" * 80 + "\n")

print("\n" + "=" * 80)
print("最终回答：")
print("=" * 80 + "\n")
print(result.final_output)


OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export



调试输出：result.new_items

ToolCallItem(agent=Agent(name='ANIFORCE Material Assistant', handoff_description=None, tools=[FunctionTool(name='get_material_thumbnail', description='获取素材缩略图。', params_json_schema={'properties': {'material_id': {'description': '素材 ID', 'title': 'Material Id', 'type': 'string'}}, 'required': ['material_id'], 'title': 'get_material_thumbnail_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x129776310>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None), FunctionTool(name='download_campaign_report', description='下载投放报告文件。', params_json_schema={'properties': {'project_id': {'description': '项目 ID', 'title': 'Project Id', 'type': 'string'}}, 'required': ['project_id'], 'title': 'down

OPENAI_API_KEY is not set, skipping trace export


In [16]:
#超时调试

import asyncio

from openai import AsyncOpenAI
from agents import Agent, ModelSettings, Runner, ToolTimeoutError, function_tool, set_tracing_disabled
from agents.models.openai_responses import OpenAIResponsesModel

MODEL = "gpt-5.3-codex"
BASE_URL = "https://api.tokenlab.sh/v1"
API_KEY = "sk-aeRemEo2sD0YgQWEFGjipWrzTp4LVFUVzHD8bD5fx5PoLMGF"

set_tracing_disabled(True)


@function_tool(timeout=1.0, timeout_behavior="error_as_result")
async def slow_material_metrics(material_id: str) -> str:
    """查询素材指标。超时后把错误作为工具结果返回给模型。"""
    await asyncio.sleep(3)
    return f"素材 {material_id}: CTR=2.3%, ROI=3.5"


@function_tool(timeout=1.0, timeout_behavior="raise_exception")
async def slow_publish_campaign(campaign_id: str) -> str:
    """发布广告计划。超时后直接让本次 run 失败。"""
    await asyncio.sleep(3)
    return f"广告计划 {campaign_id} 已发布"



client = AsyncOpenAI(api_key=API_KEY, base_url=BASE_URL, timeout=90.0, max_retries=0)
model = OpenAIResponsesModel(model=MODEL, openai_client=client)

# 1) error_as_result：工具超时会变成模型可见的工具输出，run 不崩
recover_agent = Agent(
    name="ANIFORCE Timeout Recover Agent",
    instructions="你是 ANIFORCE 助手。工具超时时，明确告诉用户超时，并给出下一步建议。",
    model=model,
    tools=[slow_material_metrics],
    model_settings=ModelSettings(parallel_tool_calls=False, truncation="auto", store=False),
)

result = await Runner.run(recover_agent, "查询素材 M001 的投放指标", max_turns=3)
print("\n=== error_as_result: result.new_items ===\n")
for item in result.new_items:
    print(item)
    print("\n" + "-" * 80 + "\n")
print("最终回答：", result.final_output)

# 2) raise_exception：工具超时会抛 ToolTimeoutError，run 失败，由业务层处理
hard_fail_agent = Agent(
    name="ANIFORCE Timeout Hard Fail Agent",
    instructions="你是 ANIFORCE 助手。发布广告计划必须调用工具。",
    model=model,
    tools=[slow_publish_campaign],
    model_settings=ModelSettings(parallel_tool_calls=False, truncation="auto", store=False),
)

print("\n=== raise_exception ===\n")
try:
    await Runner.run(hard_fail_agent, "发布广告计划 C001", max_turns=3)
except ToolTimeoutError as e:
    print(f"捕获 ToolTimeoutError: tool={e.tool_name}, timeout={e.timeout_seconds}s")


=== error_as_result: result.new_items ===

ToolCallItem(agent=Agent(name='ANIFORCE Timeout Recover Agent', handoff_description=None, tools=[FunctionTool(name='slow_material_metrics', description='查询素材指标。超时后把错误作为工具结果返回给模型。', params_json_schema={'properties': {'material_id': {'title': 'Material Id', 'type': 'string'}}, 'required': ['material_id'], 'title': 'slow_material_metrics_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x11fd7b110>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=1.0, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)], mcp_servers=[], mcp_config={}, instructions='你是 ANIFORCE 助手。工具超时时，明确告诉用户超时，并给出下一步建议。', prompt=None, handoffs=[], model=<agents.models.openai_responses.OpenAIResponsesModel object at 0x11fd7ac10>, model_settings=Model

In [17]:

import asyncio
from typing import Any, Annotated

from openai import AsyncOpenAI
from agents import Agent, ModelSettings, Runner, RunContextWrapper, function_tool, set_tracing_disabled
from agents.models.openai_responses import OpenAIResponsesModel

MODEL = "gpt-5.3-codex"
BASE_URL = "https://api.tokenlab.sh/v1"
API_KEY = "sk-aeRemEo2sD0YgQWEFGjipWrzTp4LVFUVzHD8bD5fx5PoLMGF"

set_tracing_disabled(True)


# === 1. 默认错误处理：default_tool_error_function ===

@function_tool
def flaky_meta_query(campaign_id: Annotated[str, "广告系列 ID"]) -> str:
    """查询 Meta 广告数据。可能抛异常。"""
    if campaign_id == "C001":
        return "广告系列 C001: 花费 $5000, ROI 2.8"
    raise ValueError(f"Meta API 返回错误: campaign_id={campaign_id} 不存在")


# === 2. 自定义错误函数：记录日志 + 友好提示 ===

def custom_error_handler(context: RunContextWrapper[Any], error: Exception) -> str:
    """自定义错误处理：记录日志，返回用户友好提示。"""
    print(f"[ERROR_LOG] 工具调用失败: {error}")
    return "内部服务暂时不可用，请稍后重试或联系技术支持。"


@function_tool(failure_error_function=custom_error_handler)
def flaky_material_upload(material_id: Annotated[str, "素材 ID"]) -> str:
    """上传素材到 OSS。可能失败。"""
    if material_id == "M001":
        return f"素材 {material_id} 上传成功"
    raise RuntimeError(f"OSS 上传失败: 素材 {material_id} 格式错误")


# === 3. None：错误直接抛出，由业务层处理 ===

@function_tool(failure_error_function=None)
def critical_publish_ad(ad_id: Annotated[str, "广告 ID"]) -> str:
    """发布广告（关键操作）。失败后直接抛异常。"""
    if ad_id == "AD001":
        return f"广告 {ad_id} 发布成功"
    raise PermissionError(f"权限不足：无法发布广告 {ad_id}")



client = AsyncOpenAI(api_key=API_KEY, base_url=BASE_URL, timeout=90.0, max_retries=0)
model = OpenAIResponsesModel(model=MODEL, openai_client=client)

# 1. 默认错误处理：模型看到通用错误信息
print("=== 1. 默认错误处理 ===\n")
default_agent = Agent(
    name="ANIFORCE Default Error Agent",
    instructions="你是 ANIFORCE 助手。工具失败时，告诉用户并给建议。",
    model=model,
    tools=[flaky_meta_query],
    model_settings=ModelSettings(parallel_tool_calls=False, truncation="auto", store=False),
)
result = await Runner.run(default_agent, "查询广告系列 C999 的数据", max_turns=3)
print(f"最终回答: {result.final_output}\n")

# 2. 自定义错误处理：记录日志 + 友好提示
print("=== 2. 自定义错误处理 ===\n")
custom_agent = Agent(
    name="ANIFORCE Custom Error Agent",
    instructions="你是 ANIFORCE 助手。工具失败时，告诉用户并给建议。",
    model=model,
    tools=[flaky_material_upload],
    model_settings=ModelSettings(parallel_tool_calls=False, truncation="auto", store=False),
)
result = await Runner.run(custom_agent, "上传素材 M999", max_turns=3)
print(f"最终回答: {result.final_output}\n")

# 3. None：错误直接抛出
print("=== 3. 错误直接抛出 ===\n")
critical_agent = Agent(
    name="ANIFORCE Critical Fail Agent",
    instructions="你是 ANIFORCE 助手。发布广告是关键操作。",
    model=model,
    tools=[critical_publish_ad],
    model_settings=ModelSettings(parallel_tool_calls=False, truncation="auto", store=False),
)
try:
    await Runner.run(critical_agent, "发布广告 AD999", max_turns=3)
except PermissionError as e:
    print(f"捕获 PermissionError: {e}\n")

=== 1. 默认错误处理 ===

最终回答: 查询失败：Meta API 返回 `campaign_id=C999` 不存在。  

你可以这样做：
1. **确认广告系列 ID 是否正确**（是否输错、是否缺少前缀/后缀）。  
2. **检查该系列是否已被删除或归档**。  
3. 如果你愿意，我可以帮你再查一次——请提供正确的 campaign_id。

=== 2. 自定义错误处理 ===

[ERROR_LOG] 工具调用失败: OSS 上传失败: 素材 M999 格式错误
最终回答: 上传素材 **M999** 失败了：内部服务暂时不可用。  
建议你：

1. 过几分钟后重试一次；  
2. 若多次重试仍失败，请联系技术支持并提供素材 ID：`M999`。

=== 3. 错误直接抛出 ===



UserError: Error running tool critical_publish_ad: 权限不足：无法发布广告 AD999